# 05 · Resultados y sensibilidad

Muestra la etapa (v): sensibilidades S1–S9 de la solución del Caso C (formulación §11; decisiones D-21, D-22, D-23).

Todas las cifras salen de `data/json/sensitivity.json` (`make sensitivity`, unos 6 min) y de `data/json/output.json` (`make run`). El notebook solo lee, controla que ambos archivos estén al día y exporta tablas y figuras para el informe.

- **Notación:** $x^*$ es la cartera óptima base (λ = `riskAversionLambda`).
- **Cada variante reoptimizada trae dos lecturas:**
  - la cartera reoptimizada con el modelo de la variante;
  - $x^*$ evaluada con ese mismo modelo (`basePortfolio`; en S6, `outOfSample`).

  Así se separa el efecto de cambiar la medida del efecto de reoptimizar (auditoría M-2).
- **Signos:** pérdida > 0 significa que el patrimonio cae; ΔPN > 0 significa que sube.

In [ ]:
%load_ext autoreload
%autoreload 2

import json

import numpy as np
import pandas as pd

from optimum import paths
from optimum.cleaning import load_processed
from optimum.io.json_contract import load_input
from optimum.reporting.figures import COLORS, new_figure, save_fig
from optimum.reporting.labels import SIDES, constraint_label, coupon_rule
from optimum.reporting.tables import save_table

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

doc = load_input()
out = json.loads(paths.OUTPUT_JSON.read_text(encoding="utf-8"))
sens = json.loads(paths.SENSITIVITY_JSON.read_text(encoding="utf-8"))
cfg = doc["caseParameters"]["sensitivities"]
assert sens["meta"]["config"] == cfg, "sensitivity.json desactualizado respecto de input.json: make sensitivity"
assert abs(sens["base"]["objective"] - out["objectiveValue"]) < 1e-6, "la base de sensitivity.json no es output.json"

lam, alpha = out["riskAversionLambda"], out["metrics"]["cvarAlpha"]
base = sens["base"]
base_rule = doc["caseParameters"].get("floatingCouponRule") or "FLAT"
label_opt = f"Óptimo (λ = {lam:g})"
print(f"Caso {out['caseId']} · λ = {lam} · α = {alpha} · regla de cupones base {base_rule} · "
      f"sensibilidades en {sens['meta']['runtimeSeconds']:.0f} s")
print(f"Base: objetivo {base['objective']:.3f} · E[ΔPN] − TC {base['expectedNetWorthChange'] - base['transactionCosts']:.2f} "
      f"· CVaR {base['cvarLoss']:.2f} S/ mm")


pretty = constraint_label

## Balance inicial

In [ ]:
# Ejemplo de tabla para el informe (balance inicial)
inst = load_processed("instruments")
tab = inst[["side", "name", "currency", "market_value_pen"]].assign(side=inst["side"].map(SIDES)).rename(
    columns={"side": "Lado", "name": "Posición", "currency": "Moneda", "market_value_pen": "Monto (S/ mm)"}
)
save_table(
    tab, "balance_inicial", caption="Balance inicial al 31-12-2025", label="tab:balance", fmt="{:,.1f}"
)

## S1 · Shocks de tasas (evaluación instantánea)

- **Qué se mide:** ΔPN en $t_0$ de carteras fijas, por revalorización completa con los spreads calibrados fijos (sin duración ni DV01). No se reoptimiza (D-21).
- **Shocks:**
  - paralelos de ±200 pb por moneda y en ambas;
  - `SHORT`: solo el tramo corto (ON–1Y), con transición t·s(t) en 3Y.
- **Reglas de cupones:** el óptimo se muestra también con la regla PERIOD_FORWARD, porque el tramo corto depende de D-13 (auditoría I-2).

In [ ]:
rs = pd.DataFrame(sens["rateShocks"])
names = {"initial": "Inicial", "optimal": label_opt, "meanVariance": "Media-varianza"}
order = [s["id"] for s in cfg["rateShocks"]]
s1 = (
    rs[rs["rule"] == base_rule]
    .pivot(index="shock", columns="portfolio", values="netWorth")
    .reindex(index=order, columns=list(names))
    .rename(columns=names)
)
for rule in sorted(set(rs["rule"]) - {base_rule}):
    sub = rs[(rs["rule"] == rule) & (rs["portfolio"] == "optimal")].set_index("shock")["netWorth"]
    s1[f"Óptimo, cupones {coupon_rule(rule)}"] = sub.reindex(order)
save_table(
    s1,
    "sensibilidad_tasas",
    caption=r"S1: $\Delta PN$ instantáneo en $t_0$ ante shocks de tasas (S/ mm; revalorización completa, spreads fijos). "
    r"SHORT: solo el tramo corto. Regla de cupones base: " + coupon_rule(base_rule) + ".",
    label="tab:sens-tasas",
    fmt="{:,.2f}",
)
opt_rule = rs[(rs["rule"] == base_rule) & (rs["portfolio"] == "optimal")].set_index("shock").reindex(order)
print("Descomposición del óptimo (regla base): ΔV activos, ΔV pasivos, ΔPN")
display(opt_rule[["assets", "liabilities", "netWorth"]].round(2))
s1.round(2)

In [ ]:
fig, ax = new_figure(aspect=0.45)
k = np.arange(len(s1))
cols = [c for c in s1.columns]
colors = [COLORS["neutral"], COLORS["ASSET"], COLORS["USD"], COLORS["PEN"]]
w = 0.8 / len(cols)
for j, (c, color) in enumerate(zip(cols, colors, strict=False)):
    ax.bar(k + (j - (len(cols) - 1) / 2) * w, s1[c], w, color=color, label=c, alpha=0.5 if c == "Inicial" else 1.0)
ax.axhline(0, color="black", lw=0.6)
ax.set_xticks(k, s1.index, fontsize=7)
ax.set(ylabel="ΔPN (S/ mm)", title="S1 · ΔPN instantáneo ante shocks de tasas")
ax.legend(fontsize=7, ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.12))
_ = save_fig(fig, "sensibilidad_tasas")

## S2 · Precios sombra de los límites activos

Ganancia de objetivo, en S/ mm, por cada punto porcentual que se relaja un límite activo del Cuadro 4. Se muestran tres cifras:

- **Dual del LP:** la pendiente local, con las demás restricciones fijas.
- **Diferencia finita con paso chico:** debería coincidir con el dual.
- **Diferencia finita con paso grande:** la ganancia media del tramo. El valor óptimo es cóncavo en el límite, así que da menos si la pendiente cae (auditoría M-1).

Las cotas gemelas, como pasivo fijo ≥ 45 % y flotante ≤ 55 %, son la misma restricción. Se relajan juntas en una sola fila, con la suma de sus duales (auditoría I-3).

In [ ]:
sp = pd.DataFrame(sens["shadowPrices"]).set_index("name")
small, big = 100 * cfg["limitRelaxationSmall"], 100 * cfg["limitRelaxation"]
s2 = pd.DataFrame(
    {
        "Cota": sp["bound"].str.replace("min", "mín.").str.replace("max", "máx."),
        "Límite": sp["limit"],
        "Dual": sp["dualPerPp"],
        f"Dif. finita {small:g} pp": sp["finiteDiffSmallPerPp"],
        f"Dif. finita {big:g} pp": sp["finiteDiffPerPp"],
    }
)
def twin_label(name: str) -> str:
    # 'A:B:C+A:B:D' -> 'A · B · C/D' (cotas gemelas); si no, el nombre legible
    parts = [p.split(":") for p in name.split("+")]
    if len(parts) == 2 and parts[0][:-1] == parts[1][:-1]:
        return pretty(":".join(parts[0])) + "/" + pretty(":".join(parts[1])).split(" · ")[-1]
    return " / ".join(pretty(p) for p in name.split("+"))


s2.index = [twin_label(n) for n in sp.index]
s2 = s2.sort_values("Dual", ascending=False)
assert (s2["Dual"] >= -1e-9).all()
save_table(
    s2,
    "precios_sombra",
    caption="S2: precios sombra de los límites activos (S/ mm de objetivo por 1 pp de relajación).",
    label="tab:precios-sombra",
    fmt="{:,.4f}",
)
s2

## S3–S9 · Variantes reoptimizadas

- **Qué cambia en cada variante:**
  - S3, costos: multiplicador de los costos de transacción.
  - S4, α: nivel de confianza del CVaR.
  - S5, cola: probabilidad conjunta del estrés, o se deja un estrés fuera.
  - S6, semillas: los escenarios bootstrap se generan con otra semilla.
  - S7, deriva: cambia la deriva esperada de los factores.
  - S8, estrés extra: se agrega el espejo de C_STRESS_1 con apreciación del PEN.
  - S9, cupones: regla PERIOD_FORWARD para los flotantes.
- **Columnas de "$x^*$":** la cartera óptima base evaluada con el modelo de la variante, sin reoptimizar.
- **Distancia:** $\sum_k |x_k - x^*_k|$, en S/ mm.
- **CVaR en S4:** cada fila usa su propio α, así que esas cifras no son comparables con el CVaR al 95 %.

In [ ]:
def pct_w(row, dim, side, cat):
    return None if row["weights"] is None else 100 * row["weights"][dim][side].get(cat, 0.0)


def net(m: dict) -> float:
    return m["expectedNetWorthChange"] - m["transactionCosts"]


def summary(group: str, label: str, row: dict) -> dict:
    ev = row.get("basePortfolio") or row.get("outOfSample")
    ok = row["status"] in ("OPTIMAL", "OPTIMAL_INACCURATE")
    return {
        "Grupo": group,
        "Variante": label,
        "Status": row["status"],
        "E − TC": net(row) if ok else None,
        "CVaR": row["cvarLoss"],
        "Rotación": row["turnover"],
        "Distancia L1 a x*": row["distanceToBase"],
        "Activos USD (%)": pct_w(row, "byCurrency", "ASSET", "USD"),
        "Pasivos flot. (%)": pct_w(row, "byRateType", "LIABILITY", "FLOAT"),
        "E − TC de x*": net(ev) if ev else None,
        "CVaR de x*": ev["cvarLoss"] if ev else None,
    }


assert "turnover" in base, "sensitivity.json sin turnover en la base: make sensitivity"
rows = [summary("Base", "Base", {**base, "distanceToBase": 0.0, "basePortfolio": base})]
rows += [summary("S3 costos", f"TC × {r['multiplier']:g}", r) for r in sens["costs"]]
rows += [summary("S4 α", f"α = {r['alpha']:g}", r) for r in sens["alphas"]]
rows += [summary("S5 cola", f"P estrés = {r['stressProbability']:g}", r) for r in sens["stressWeight"]]
rows += [summary("S5 cola", f"sin {r['dropped']}", r) for r in sens["stressLeaveOneOut"]]
rows += [summary("S6 semillas", f"semilla {r['seed']}", r) for r in sens["seeds"]]
rows += [summary("S7 deriva", "deriva nula" if r["variant"] == "ALL_ZERO" else f"deriva {r['variant']}", r) for r in sens["drift"]]
rows += [summary("S8 estrés", f"+ {r['scenario']}", r) for r in sens["extraStress"]]
rows += [summary("S9 cupones", f"cupones {coupon_rule(r['rule'])}", r) for r in sens["floatingCouponRule"]]
s39 = pd.DataFrame(rows).set_index("Variante")
print("Variantes no óptimas:", list(s39.index[~s39["Status"].isin(["OPTIMAL", "OPTIMAL_INACCURATE"])]) or "ninguna")
save_table(
    s39.drop(columns=["Status"]),
    "sensibilidades_resumen",
    caption=r"S3--S9: variantes reoptimizadas y cartera base $x^*$ evaluada con el modelo de cada variante (S/ mm y \% del lado). "
    r"En S4 el CVaR usa el $\alpha$ de la fila. Rotación: $\sum_k |z_k - z_k^0|$; distancia: $\sum_k |z_k - z_k^*|$ (S/ mm).",
    label="tab:sens-resumen",
    fmt="{:,.2f}",
)
s39.round(2)

In [ ]:
# Robustez del CVaR: x* evaluada con cada modelo vs. la cartera reoptimizada (misma α; se omiten S3 y S4)
fig_rows = s39[~s39["Grupo"].isin(["S3 costos", "S4 α"])].iloc[::-1]
fig, ax = new_figure(aspect=0.62)
k = np.arange(len(fig_rows))
ax.barh(k + 0.2, fig_rows["CVaR de x*"], 0.4, color=COLORS["neutral"], alpha=0.6, label="x* sin reoptimizar")
ax.barh(k - 0.2, fig_rows["CVaR"], 0.4, color=COLORS["ASSET"], label="Reoptimizada")
ax.axvline(base["cvarLoss"], color="black", ls=":", lw=1, label=f"CVaR base ({base['cvarLoss']:.1f})")
ax.set_yticks(k, fig_rows.index, fontsize=7)
ax.set(xlabel=f"CVaR {alpha * 100:g} % de la pérdida (S/ mm)", title="Robustez del CVaR ante cambios de modelo")
ax.legend(fontsize=7, loc="lower right")
_ = save_fig(fig, "robustez_cvar")

## S9 · Estrés según la regla de cupones flotantes

Pérdida en cada escenario de estrés, con tres combinaciones:

- $x^*$ con la regla base;
- $x^*$ con la regla alternativa;
- la cartera reoptimizada con la regla alternativa.

Así queda precisada la cautela del notebook 04 (auditoría I-1, D-23).

In [ ]:
s9 = pd.DataFrame({f"x*, {coupon_rule(base_rule)}": pd.Series(base["stressLosses"])})
for r in sens["floatingCouponRule"]:
    s9[f"x*, {coupon_rule(r['rule'])}"] = pd.Series(r["basePortfolio"]["stressLosses"])
    s9[f"Reopt., {coupon_rule(r['rule'])}"] = pd.Series(r["stressLosses"])
extra = sens["extraStress"]
save_table(
    s9,
    "estres_regla_cupones",
    caption=r"S9: pérdida en cada estrés (S/ mm; $<0$ = ganancia) según la regla de cupones flotantes.",
    label="tab:estres-cupones",
    fmt="{:,.2f}",
)
for r in extra:
    print(f"S8 · {r['scenario']}: x* pierde {r['basePortfolio']['stressLosses'][r['scenario']]:.1f}; "
          f"la reoptimizada, {r['stressLosses'][r['scenario']]:.1f} S/ mm")
s9.round(2)

## Lectura

> Borrador redactado por la IA a partir de las salidas de este notebook y de la auditoría de sensibilidades (D-23). Lo revisa y valida el alumno.

**1. Tasas (S1).** El óptimo neutraliza el riesgo de tasas PEN pero toma riesgo de tasas USD.
- **Cartera inicial:** gana 17.3 S/ mm con PEN +200 pb y pierde 21.4 con PEN −200 pb.
- **Óptimo:** con esos mismos shocks queda en −2.5 y +0.5. En cambio gana 12.5 con USD +200 pb y pierde 14.1 con USD −200 pb.
- **Por qué:** sale A04 (bono USD fijo) y queda L05 (bono USD fijo en el pasivo, 140), sin nada que lo compense. Por eso la pérdida en C_STRESS_2 (caída de tasas) es la que define la cola. Media-varianza tiene casi la misma exposición.
- **Tramo corto:** el +8.5 de SHORT+200 no es confiable. Con la regla de cupones PERIOD_FORWARD es −0.5 (D-23, I-2). Los shocks paralelos no cambian con la regla.

**2. Precios sombra (S2).** Los límites que más valen son renta fija ≤ 75 % (0.38–0.39 S/ mm por pp) y caja ≤ 20 % (0.19).
- **Lectura conjunta:** ambos implican un piso de 5 % de equity. Su precio es en buena parte el costo de no poder bajar más el equity.
- **Magnitud:** relajar la renta fija 5 pp sube el objetivo en 1.7 S/ mm, cuando el objetivo base es 1.16.
- **Resto:** A02 vale 0.09 por pp, L01 0.02 y el par fijo/flotante 0.01.
- **Consistencia:** el dual coincide con la diferencia finita de 0.5 pp. La de 5 pp es menor porque la pendiente cae.

**3. Costos (S3).** La solución depende mucho del supuesto de costos (D-08).
- **Sin costos:** rota 1,473 S/ mm y el CVaR baja a 3.2.
- **Con costos ×2:** el CVaR sube a 8.6.
- **Con costos ×4:** $x^*$ tendría E − TC = −5.1, peor que no hacer nada (3.96).

**4. Nivel de confianza (S4).** El CVaR sube con α (8.1 con 0.975; 9.2 con 0.99), pero las posiciones apenas cambian (distancia 37–60 S/ mm): la solución es estable ante α.

**5. Cola de estrés (S5).**
- **C_STRESS_2 es el único estrés que restringe la solución.** Sin él, la cartera reoptimizada reduce los activos USD al mínimo de 20 % y su CVaR baja a 0.5, aunque $x^*$ sin reoptimizar sigue en 6.4.
- **Dejar fuera otro estrés sube el CVaR de $x^*$ a 8.2.** No es un efecto de ese estrés: al repartir su probabilidad, C_STRESS_2 pasa de 1.25 % a 1.67 %.
- **Con P = 20 % el optimizador apuesta por los estrés.** Como todos deprecian el PEN, sube los activos USD a 45 % y E − TC a 9.3, pero el CVaR llega a 20.5.

**6. Muestreo (S6).**
- **Fuera de muestra:** el CVaR de $x^*$ va de 7.3 a 7.9, contra 7.8 en muestra, y E − TC de 2.8 a 3.3. No hay señal de sobreajuste al bootstrap.
- **Posiciones al reoptimizar:** se mueven entre 17 y 134 S/ mm.
- **Límite de la prueba:** los estrés son los mismos en todas las semillas, así que no mide sobreajuste respecto de ellos (M-3).

**7. Deriva (S7).** Si se anula toda deriva, E − TC de $x^*$ cae de 3.11 a −0.23 y su CVaR sube a 9.9. Casi todo el resultado esperado viene de la deriva supuesta de FX (paridad de tasas) y de renta variable (media histórica), D-06. Con deriva cero, el carry menos los costos deja E − TC en torno a 0.

**8. Apreciación del PEN (S8).** $x^*$ pierde 21.1 S/ mm en el estrés espejo y su CVaR sube a 11.7. Reoptimizada con ese estrés, la cartera baja los activos USD a 20 %, pierde solo 1.0 en él y su CVaR queda en 7.3. La posición larga en USD de $x^*$ responde al conjunto de estrés dado; no es una cobertura neutral.

**9. Cupones flotantes (S9).**
- **C_STRESS_3:** la ganancia de $x^*$ cae de 18.8 a 5.8, y a 0.9 si se reoptimiza.
- **C_STRESS_4:** es robusta (23.2 y 15.4).
- **CVaR:** el de $x^*$ casi no cambia (8.1 frente a 7.8), así que el ranking de riesgo no depende de la regla.

**Conclusión.**
- **Robusto:** la cobertura de tasas PEN, las posiciones de activos PEN y un CVaR de 7–8 S/ mm ante semillas, α y regla de cupones.
- **Frágil:** el posicionamiento en USD, tanto en tasas como en FX, depende del conjunto de estrés (S5, S8). El resultado esperado depende de la deriva (S7) y de los costos (S3). Hay que decirlo en la opinión del analista.

> Pendiente de validación del alumno.